In [ ]:
!pip install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q sentence-transformers --no-deps
!pip install -q transformers tokenizers huggingface-hub safetensors tqdm scikit-learn scipy pillow
!pip install -q rank_bm25 ir_measures


In [1]:
!nvidia-smi

Sun May  3 11:20:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             28W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
os.listdir('/kaggle/input/datasets/smakov/search-dataset/search-dataset')

['documents.csv', 'mirage', 'wikIR1k']

In [8]:
import os, re, json, math, time, random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader

from rank_bm25 import BM25Okapi
import ir_measures
from ir_measures import P, MAP, nDCG, read_trec_qrels

RNG_SEED = 42
np.random.seed(RNG_SEED)
random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

WIKI_DIR   = '/kaggle/input/datasets/smakov/search-dataset/search-dataset/wikIR1k'
DOCS_CSV   = '/kaggle/input/datasets/smakov/search-dataset/search-dataset/documents.csv'
MIRAGE_DIR = '/kaggle/input/datasets/smakov/search-dataset/search-dataset/mirage'


device: cuda


## Загрузка данных


In [9]:
def load_docs(path):
    ids, docs = [], []
    with open(path, encoding='utf-8') as f:
        next(f)
        for line in f:
            i = line.find(',')
            ids.append(line[:i])
            docs.append(line[i+1:].rstrip('\n').split())
    return ids, docs

def load_queries(path):
    df = pd.read_csv(path)
    return {str(i): t for i, t in zip(df['id_left'], df['text_left'])}

def load_qrels_dict(path):
    out = defaultdict(dict)
    with open(path) as f:
        for line in f:
            qid, _, did, rel = line.split()
            out[qid][did] = int(rel)
    return dict(out)

doc_ids, docs_orig = load_docs(DOCS_CSV)
doc_id_to_pos = {d: i for i, d in enumerate(doc_ids)}

train_queries = load_queries(f'{WIKI_DIR}/training/queries.csv')
test_queries  = load_queries(f'{WIKI_DIR}/test/queries.csv')
train_qrels   = load_qrels_dict(f'{WIKI_DIR}/training/qrels')
test_qrels    = load_qrels_dict(f'{WIKI_DIR}/test/qrels')
test_qrels_trec  = list(read_trec_qrels(f'{WIKI_DIR}/test/qrels'))
train_qrels_trec = list(read_trec_qrels(f'{WIKI_DIR}/training/qrels'))

print(f'docs: {len(doc_ids):,}  train queries: {len(train_queries)}  test queries: {len(test_queries)}')


docs: 369,721  train queries: 1444  test queries: 100


In [ ]:
with open(f'{MIRAGE_DIR}/dataset.json') as f:
    mirage_dataset = json.load(f)
with open(f'{MIRAGE_DIR}/doc_pool.json') as f:
    mirage_pool = json.load(f)
with open(f'{MIRAGE_DIR}/oracle.json') as f:
    mirage_oracle = json.load(f)

oracle_chunk_by_qid = {k: v['doc_chunk'] for k, v in mirage_oracle.items()}

# Глобальные позиции кандидатов каждого запроса в mirage_pool
pool_idx_by_qid = defaultdict(list)
for i, ch in enumerate(mirage_pool):
    pool_idx_by_qid[ch['mapped_id']].append(i)

# Стратифицированный split 80/20 по source
random.seed(RNG_SEED)
dataset_by_src = defaultdict(list)
for d in mirage_dataset:
    dataset_by_src[d['source']].append(d)

train_queries_m, test_queries_m = [], []
for src, qs in dataset_by_src.items():
    idx = list(range(len(qs)))
    random.shuffle(idx)
    cut = int(len(idx) * 0.8)
    train_queries_m.extend(qs[i] for i in idx[:cut])
    test_queries_m.extend(qs[i] for i in idx[cut:])

# Qrels для MIRAGE test: оракульный чанк = 1, остальные = 0
mirage_test_qrels = []
for q in test_queries_m:
    qid = q['query_id']
    oracle_text = oracle_chunk_by_qid.get(qid, '')
    for idx in pool_idx_by_qid[qid]:
        rel = 1 if mirage_pool[idx]['doc_chunk'] == oracle_text else 0
        mirage_test_qrels.append(ir_measures.Qrel(qid, str(idx), rel))

print(f'MIRAGE: всего {len(mirage_dataset)} запросов, pool {len(mirage_pool)} чанков')
print(f'train: {len(train_queries_m)}, test: {len(test_queries_m)}')


MIRAGE: всего 7560 запросов, pool 37800 чанков
train: 6047, test: 1513


## BM25 бейзлайн


In [11]:
MEASURES        = [P@1, P@10, P@20, MAP, nDCG@20]
MIRAGE_MEASURES = [P@1, nDCG@5, MAP]
TOP_K = 1000

t0 = time.time()
bm25 = BM25Okapi(docs_orig)
print(f'BM25 WikiIR построен за {time.time()-t0:.1f}s')

def bm25_topk(q_tokens, k=TOP_K):
    scores = bm25.get_scores(q_tokens)
    k = min(k, len(scores))
    idx = np.argpartition(-scores, k - 1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    return idx, scores[idx]

bm25_run = []
for qid, raw in tqdm(test_queries.items(), desc='BM25 WikiIR test'):
    idx, sc = bm25_topk(raw.split())
    for i, s in zip(idx, sc):
        bm25_run.append(ir_measures.ScoredDoc(qid, doc_ids[i], float(s)))

bm25_wiki_metrics = ir_measures.calc_aggregate(MEASURES, test_qrels_trec, bm25_run)
print('BM25 WikiIR test:')
for m in MEASURES:
    print(f'  {str(m):<12}{bm25_wiki_metrics[m]:.4f}')


BM25 WikiIR построен за 22.6s


BM25 WikiIR test:   0%|          | 0/100 [00:00<?, ?it/s]

BM25 WikiIR test:
  P@1         0.4900
  P@10        0.2120
  P@20        0.1500
  AP          0.1752
  nDCG@20     0.3570


In [12]:
# BM25 на всех чанках MIRAGE; для каждого запроса ранжируем его 5 кандидатов
tok_chunks = [ch['doc_chunk'].lower().split() for ch in mirage_pool]
bm25_mirage = BM25Okapi(tok_chunks)
print(f'BM25 MIRAGE построен на {len(tok_chunks)} чанках')

def mirage_bm25_rank(q_text, qid):
    q_tokens = q_text.lower().split()
    all_scores = bm25_mirage.get_scores(q_tokens)
    cand_idxs = pool_idx_by_qid[qid]
    return sorted([(idx, float(all_scores[idx])) for idx in cand_idxs], key=lambda x: -x[1])

mirage_bm25_run = []
for q in test_queries_m:
    qid, qtext = q['query_id'], q['query']
    for idx, score in mirage_bm25_rank(qtext, qid):
        mirage_bm25_run.append(ir_measures.ScoredDoc(qid, str(idx), score))

bm25_mirage_metrics = ir_measures.calc_aggregate(MIRAGE_MEASURES, mirage_test_qrels, mirage_bm25_run)
print('BM25 MIRAGE test:')
for m in MIRAGE_MEASURES:
    print(f'  {str(m):<12}{bm25_mirage_metrics[m]:.4f}')


BM25 MIRAGE построен на 37800 чанках
BM25 MIRAGE test:
  P@1         0.5288
  nDCG@5      0.7769
  AP          0.7029


## Задание 1. Ранжирование с bi-encoders

Выбор моделей:

all-MiniLM-L6-v2 —  84 dim, обучена на 1B пар предложений. Хорошее соотношение скорость/качество, стандартный baseline для dense ретривал

all-mpnet-base-v2 — 768 dim, обучен на 1B пар с улучшенной функцией потерь. Показывает наилучшее качество среди общих bi-encoder моделей по бенчмаркам SBERT

In [13]:
MODEL1_NAME = 'all-MiniLM-L6-v2'
MODEL2_NAME = 'all-mpnet-base-v2'

model_mini  = SentenceTransformer(MODEL1_NAME, device=DEVICE)
model_mpnet = SentenceTransformer(MODEL2_NAME, device=DEVICE)
print('Модели загружены')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Модели загружены


In [14]:
doc_texts = [' '.join(d) for d in docs_orig]
n_docs = len(doc_texts)

print(f'Кодируем {n_docs:,} документов WikiIR (all-MiniLM-L6-v2)...')
t0 = time.time()
corpus_emb_mini = model_mini.encode(
    doc_texts, batch_size=256, show_progress_bar=True, normalize_embeddings=True
)  # numpy array на CPU
print(f'  готово за {time.time()-t0:.1f}s, shape: {corpus_emb_mini.shape}')
torch.cuda.empty_cache()

print(f'Кодируем {n_docs:,} документов WikiIR (all-mpnet-base-v2)...')
t0 = time.time()
corpus_emb_mpnet = model_mpnet.encode(
    doc_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True
)  # numpy array на CPU
print(f'  готово за {time.time()-t0:.1f}s, shape: {corpus_emb_mpnet.shape}')
torch.cuda.empty_cache()

Кодируем 369,721 документов WikiIR (all-MiniLM-L6-v2)...


Batches:   0%|          | 0/1445 [00:00<?, ?it/s]

  готово за 651.8s, shape: (369721, 384)
Кодируем 369,721 документов WikiIR (all-mpnet-base-v2)...


Batches:   0%|          | 0/5777 [00:00<?, ?it/s]

  готово за 3511.6s, shape: (369721, 768)


In [ ]:
def biencoder_retrieve_wiki(model, corpus_emb, top_k=1000):
    q_texts = list(test_queries.values())
    q_ids   = list(test_queries.keys())
    q_emb = model.encode(q_texts, batch_size=128, normalize_embeddings=True)  # numpy
    scores_mat = q_emb @ corpus_emb.T  # (n_queries, n_docs)
    run = []
    for qi, qid in enumerate(q_ids):
        row = scores_mat[qi]
        k = min(top_k, len(row))
        top_idx = np.argpartition(-row, k - 1)[:k]
        top_idx = top_idx[np.argsort(-row[top_idx])]
        for i in top_idx:
            run.append(ir_measures.ScoredDoc(qid, doc_ids[i], float(row[i])))
    return run

run_mini  = biencoder_retrieve_wiki(model_mini,  corpus_emb_mini)
run_mpnet = biencoder_retrieve_wiki(model_mpnet, corpus_emb_mpnet)

m_mini  = ir_measures.calc_aggregate(MEASURES, test_qrels_trec, run_mini)
m_mpnet = ir_measures.calc_aggregate(MEASURES, test_qrels_trec, run_mpnet)

cmp_task1_wiki = pd.DataFrame({
    'BM25 (baseline)':   {str(m): bm25_wiki_metrics[m]  for m in MEASURES},
    'all-MiniLM-L6-v2': {str(m): m_mini[m]  for m in MEASURES},
    'all-mpnet-base-v2': {str(m): m_mpnet[m] for m in MEASURES},
})
print('WikiIR test:')
print(cmp_task1_wiki.round(4).to_string())

WikiIR test:
         BM25 (baseline)  all-MiniLM-L6-v2  all-mpnet-base-v2
P@1               0.4900            0.6400             0.7800
P@10              0.2120            0.1850             0.2080
P@20              0.1500            0.1265             0.1360
AP                0.1752            0.1397             0.1713
nDCG@20           0.3570            0.3574             0.4024


In [ ]:
chunk_texts = [ch['doc_chunk'] for ch in mirage_pool]

print('Кодируем MIRAGE chunks (all-MiniLM-L6-v2)...')
chunk_emb_mini = model_mini.encode(
    chunk_texts, batch_size=512, show_progress_bar=True,
    convert_to_tensor=True, device=DEVICE, normalize_embeddings=True
)
print('Кодируем MIRAGE chunks (all-mpnet-base-v2)...')
chunk_emb_mpnet = model_mpnet.encode(
    chunk_texts, batch_size=256, show_progress_bar=True,
    convert_to_tensor=True, device=DEVICE, normalize_embeddings=True
)
print(f'chunk_emb_mini: {tuple(chunk_emb_mini.shape)}, chunk_emb_mpnet: {tuple(chunk_emb_mpnet.shape)}')


Кодируем MIRAGE chunks (all-MiniLM-L6-v2)...


Batches:   0%|          | 0/74 [00:00<?, ?it/s]

Кодируем MIRAGE chunks (all-mpnet-base-v2)...


Batches:   0%|          | 0/148 [00:00<?, ?it/s]

chunk_emb_mini: (37800, 384), chunk_emb_mpnet: (37800, 768)


In [ ]:
def biencoder_rank_mirage(model, chunk_emb):
    q_texts = [q['query'] for q in test_queries_m]
    q_ids   = [q['query_id'] for q in test_queries_m]
    q_emb = model.encode(q_texts, batch_size=128, convert_to_tensor=True,
                          device=DEVICE, normalize_embeddings=True)
    run = []
    for qid, qe in zip(q_ids, q_emb):
        cand_idxs = pool_idx_by_qid[qid]
        cand_emb  = chunk_emb[cand_idxs]
        scores    = (cand_emb @ qe).cpu().numpy()
        for idx, sc in zip(cand_idxs, scores):
            run.append(ir_measures.ScoredDoc(qid, str(idx), float(sc)))
    return run

run_m_mini  = biencoder_rank_mirage(model_mini,  chunk_emb_mini)
run_m_mpnet = biencoder_rank_mirage(model_mpnet, chunk_emb_mpnet)

mm_mini  = ir_measures.calc_aggregate(MIRAGE_MEASURES, mirage_test_qrels, run_m_mini)
mm_mpnet = ir_measures.calc_aggregate(MIRAGE_MEASURES, mirage_test_qrels, run_m_mpnet)

cmp_task1_mirage = pd.DataFrame({
    'BM25 (baseline)':   {str(m): bm25_mirage_metrics[m] for m in MIRAGE_MEASURES},
    'all-MiniLM-L6-v2': {str(m): mm_mini[m]  for m in MIRAGE_MEASURES},
    'all-mpnet-base-v2': {str(m): mm_mpnet[m] for m in MIRAGE_MEASURES},
})
print('MIRAGE test:')
print(cmp_task1_mirage.round(4).to_string())


MIRAGE test:
        BM25 (baseline)  all-MiniLM-L6-v2  all-mpnet-base-v2
P@1              0.5288            0.6200             0.6576
nDCG@5           0.7769            0.8326             0.8509
AP               0.7029            0.7759             0.8002



WikiIR: bi-encoders значительно улучшают P@1 — MiniLM даёт 0.64 (+15 над BM25 0.49). nDCG@20 растёт меньше: MiniLM 0.357 (на уровне BM25), mpnet 0.402 (+4.5). Это характерно для биэнкодеров - хуже ранжируют длинный хвост, чем BM25 с точным совпадением. mpnet стабильно лучше MiniLM на всех метриках благодаря вдвое большей размерности эмбеддинга

MIRAGE: mpnet P@1 = 0.658 (+12.9 над BM25), nDCG@5 = 0.851. MiniLM P@1 = 0.620. Оба биэнкодера существенно превосходят BM25. Для дальнейших экспериментов используем mpnet как более сильную модель

## Задание 2. Реранжирование cross-encoder

Выбор модели: cross-encoder/ms-marco-MiniLM-L-6-v2 — 6-слойный MiniLM cross-encoder, дообученный на MS MARCO passage ranking. Компактный (66 MB), быстрее MS-MARCO-MiniLM-L-12-v2 примерно в 2 раза при сопоставимом качестве. Требует O(k) инференсов на запрос

In [18]:
CE_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
cross_encoder = CrossEncoder(CE_NAME, device=DEVICE, max_length=512)
print(f'Cross-encoder {CE_NAME} загружен')


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-encoder cross-encoder/ms-marco-MiniLM-L-6-v2 загружен


In [19]:
def rerank_wiki_top_k(ce, k):
    run = []
    t_bm25_total, t_ce_total = 0.0, 0.0
    for qid, raw in tqdm(test_queries.items(), desc=f'CE k={k}'):
        t0 = time.time()
        idx, _ = bm25_topk(raw.split(), k=k)
        t_bm25_total += time.time() - t0

        pairs = [(raw, ' '.join(docs_orig[i])) for i in idx]
        t0 = time.time()
        scores = ce.predict(pairs, show_progress_bar=False)
        t_ce_total += time.time() - t0

        for i, sc in zip(idx, scores):
            run.append(ir_measures.ScoredDoc(qid, doc_ids[i], float(sc)))
    n_q = len(test_queries)
    return run, t_bm25_total / n_q * 1000, t_ce_total / n_q * 1000

rerank_results = {}
for k in [10, 50, 100]:
    run, ms_bm25, ms_ce = rerank_wiki_top_k(cross_encoder, k)
    metrics = ir_measures.calc_aggregate(MEASURES, test_qrels_trec, run)
    rerank_results[k] = {'metrics': metrics, 'ms_bm25': ms_bm25, 'ms_ce': ms_ce}
    print(f'k={k:3d}: nDCG@20={metrics[nDCG@20]:.4f}  P@1={metrics[P@1]:.4f}  '
          f'BM25 {ms_bm25:.1f}ms/q  CE {ms_ce:.1f}ms/q')

CE k=10:   0%|          | 0/100 [00:00<?, ?it/s]

k= 10: nDCG@20=0.3759  P@1=0.7500  BM25 369.7ms/q  CE 24.2ms/q


CE k=50:   0%|          | 0/100 [00:00<?, ?it/s]

k= 50: nDCG@20=0.4469  P@1=0.8100  BM25 369.7ms/q  CE 86.3ms/q


CE k=100:   0%|          | 0/100 [00:00<?, ?it/s]

k=100: nDCG@20=0.4487  P@1=0.8000  BM25 369.4ms/q  CE 149.6ms/q


In [20]:
cmp_task2_wiki = pd.DataFrame({
    'BM25 (baseline)': {str(m): bm25_wiki_metrics[m] for m in MEASURES},
    **{f'CE k={k}':    {str(m): rerank_results[k]['metrics'][m] for m in MEASURES}
       for k in [10, 50, 100]},
})
print('Задание 2 — WikiIR test:')
print(cmp_task2_wiki.round(4).to_string())

timing = pd.DataFrame({
    f'k={k}': {'BM25 ms/q': rerank_results[k]['ms_bm25'],
               'CE ms/q':   rerank_results[k]['ms_ce'],
               'total ms/q': rerank_results[k]['ms_bm25'] + rerank_results[k]['ms_ce']}
    for k in [10, 50, 100]
}).T
print('\nЭффективность реранжирования (ms на запрос):')
print(timing.round(1).to_string())


Задание 2 — WikiIR test:
         BM25 (baseline)  CE k=10  CE k=50  CE k=100
P@1               0.4900   0.7500   0.8100    0.8000
P@10              0.2120   0.2150   0.2350    0.2320
P@20              0.1500   0.1075   0.1625    0.1665
AP                0.1752   0.1540   0.1993    0.2050
nDCG@20           0.3570   0.3759   0.4469    0.4487

Эффективность реранжирования (ms на запрос):
       BM25 ms/q  CE ms/q  total ms/q
k=10       369.7     24.2       393.9
k=50       369.7     86.3       456.0
k=100      369.4    149.6       519.0


In [21]:
def rerank_mirage_ce(ce, k=5):
    run = []
    for q in tqdm(test_queries_m, desc=f'MIRAGE CE k={k}'):
        qid, qtext = q['query_id'], q['query']
        ranked = mirage_bm25_rank(qtext, qid)[:k]
        pairs  = [(qtext, mirage_pool[idx]['doc_chunk']) for idx, _ in ranked]
        scores = ce.predict(pairs, show_progress_bar=False)
        for (idx, _), sc in zip(ranked, scores):
            run.append(ir_measures.ScoredDoc(qid, str(idx), float(sc)))
    return run

run_ce_mirage = rerank_mirage_ce(cross_encoder, k=5)
mm_ce = ir_measures.calc_aggregate(MIRAGE_MEASURES, mirage_test_qrels, run_ce_mirage)

cmp_task2_mirage = pd.DataFrame({
    'BM25 (baseline)': {str(m): bm25_mirage_metrics[m] for m in MIRAGE_MEASURES},
    'CE k=5':          {str(m): mm_ce[m] for m in MIRAGE_MEASURES},
})
print('Задание 2 — MIRAGE test:')
print(cmp_task2_mirage.round(4).to_string())


MIRAGE CE k=5:   0%|          | 0/1513 [00:00<?, ?it/s]

Задание 2 — MIRAGE test:
        BM25 (baseline)  CE k=5
P@1              0.5288  0.8057
nDCG@5           0.7769  0.9173
AP               0.7029  0.8891


WikiIR: реранжирование значительно улучшает все метрики. k=10 уже даёт P@1 = 0.75 (+26 над BM25); k=50 — лучший P@1 = 0.81 (+32) и nDCG@20 = 0.447 (+9); k=100 — нDCG@20 = 0.449, лучший из всех k по этой метрике

Эффективность: BM25 занимает ~370 ms/q (полный проход по 370k документов). Кросс-энкодер добавляет 24 ms/q при k=10 и 150 ms/q при k=100. Оптимальный компромисс — k=50: лучший P@1 при вдвое меньшей CE-задержке по сравнению с k=100

MIRAGE: CE k=5 даёт P@1 = 0.806 (+27.7 над BM25), nDCG@5 = 0.917 (+14)

## Задание 3. Смешанная модель


In [22]:
TOP_K_MIX = 100

# Кодируем train-запросы WikiIR (numpy на CPU)
train_q_ids   = list(train_queries.keys())
train_q_texts = [train_queries[qid] for qid in train_q_ids]
train_q_emb   = model_mpnet.encode(
    train_q_texts, batch_size=64, normalize_embeddings=True
)  # numpy

# Предвычисляем нормализованные BM25 и косинус для top-100 кандидатов
tr_bm25_idx  = {}
tr_bm25_norm = {}
tr_cos_norm  = {}

for qid, qe in tqdm(zip(train_q_ids, train_q_emb), total=len(train_q_ids), desc='precompute train'):
    idx, bm25_sc = bm25_topk(train_queries[qid].split(), k=TOP_K_MIX)
    bm25_n = (bm25_sc - bm25_sc.min()) / (bm25_sc.max() - bm25_sc.min() + 1e-9)
    cand_emb = corpus_emb_mpnet[idx.tolist()]  # numpy slice
    cos_sc   = cand_emb @ qe                   # numpy matmul
    cos_n    = (cos_sc + 1.0) / 2.0
    tr_bm25_idx[qid]  = idx
    tr_bm25_norm[qid] = bm25_n
    tr_cos_norm[qid]  = cos_n

print('Завершено')

precompute train:   0%|          | 0/1444 [00:00<?, ?it/s]

Завершено


In [23]:
alpha_grid = np.arange(0.0, 1.05, 0.05)
best_alpha, best_ndcg = 0.5, -1.0
alpha_scores = []

for alpha in tqdm(alpha_grid, desc='alpha grid'):
    run = []
    for qid in train_q_ids:
        mix_sc = alpha * tr_bm25_norm[qid] + (1.0 - alpha) * tr_cos_norm[qid]
        for i, sc in zip(tr_bm25_idx[qid], mix_sc):
            run.append(ir_measures.ScoredDoc(qid, doc_ids[i], float(sc)))
    score = ir_measures.calc_aggregate([nDCG@20], train_qrels_trec, run)[nDCG@20]
    alpha_scores.append((float(alpha), score))
    if score > best_ndcg:
        best_ndcg, best_alpha = score, float(alpha)

print(f'Лучший alpha = {best_alpha:.2f},  nDCG@20 на train = {best_ndcg:.4f}')
print('\nalpha vs nDCG@20 (train):')
for a, s in alpha_scores:
    marker = ' <-- best' if abs(a - best_alpha) < 0.001 else ''
    print(f'  alpha={a:.2f}  nDCG@20={s:.4f}{marker}')

alpha grid:   0%|          | 0/21 [00:00<?, ?it/s]

Лучший alpha = 0.05,  nDCG@20 на train = 0.4533

alpha vs nDCG@20 (train):
  alpha=0.00  nDCG@20=0.4411
  alpha=0.05  nDCG@20=0.4533 <-- best
  alpha=0.10  nDCG@20=0.4529
  alpha=0.15  nDCG@20=0.4450
  alpha=0.20  nDCG@20=0.4357
  alpha=0.25  nDCG@20=0.4268
  alpha=0.30  nDCG@20=0.4183
  alpha=0.35  nDCG@20=0.4119
  alpha=0.40  nDCG@20=0.4058
  alpha=0.45  nDCG@20=0.3995
  alpha=0.50  nDCG@20=0.3955
  alpha=0.55  nDCG@20=0.3913
  alpha=0.60  nDCG@20=0.3879
  alpha=0.65  nDCG@20=0.3839
  alpha=0.70  nDCG@20=0.3815
  alpha=0.75  nDCG@20=0.3792
  alpha=0.80  nDCG@20=0.3770
  alpha=0.85  nDCG@20=0.3749
  alpha=0.90  nDCG@20=0.3738
  alpha=0.95  nDCG@20=0.3722
  alpha=1.00  nDCG@20=0.3612


In [24]:
# Применяем смешанную модель на WikiIR test (numpy на CPU)
test_q_ids   = list(test_queries.keys())
test_q_texts = [test_queries[qid] for qid in test_q_ids]
test_q_emb   = model_mpnet.encode(
    test_q_texts, batch_size=64, normalize_embeddings=True
)  # numpy

mix_wiki_run = []
for qid, qe in zip(test_q_ids, test_q_emb):
    idx, bm25_sc = bm25_topk(test_queries[qid].split(), k=TOP_K_MIX)
    bm25_n   = (bm25_sc - bm25_sc.min()) / (bm25_sc.max() - bm25_sc.min() + 1e-9)
    cand_emb = corpus_emb_mpnet[idx.tolist()]  # numpy slice
    cos_n    = ((cand_emb @ qe) + 1.0) / 2.0   # numpy matmul
    mix_sc   = best_alpha * bm25_n + (1.0 - best_alpha) * cos_n
    for i, sc in zip(idx, mix_sc):
        mix_wiki_run.append(ir_measures.ScoredDoc(qid, doc_ids[i], float(sc)))

m_mix_wiki = ir_measures.calc_aggregate(MEASURES, test_qrels_trec, mix_wiki_run)
print(f'Mixture (alpha={best_alpha:.2f}) WikiIR test:')
for m in MEASURES:
    print(f'  {str(m):<12}{m_mix_wiki[m]:.4f}')

Mixture (alpha=0.05) WikiIR test:
  P@1         0.7700
  P@10        0.2650
  P@20        0.1725
  AP          0.2226
  nDCG@20     0.4598


In [25]:
# Применяем смешанную модель на MIRAGE test
mirage_q_texts = [q['query'] for q in test_queries_m]
mirage_q_ids   = [q['query_id'] for q in test_queries_m]
mirage_q_emb   = model_mpnet.encode(
    mirage_q_texts, batch_size=128, convert_to_tensor=True,
    device=DEVICE, normalize_embeddings=True
)

mix_mirage_run = []
for q, qe in zip(test_queries_m, mirage_q_emb):
    qid, qtext = q['query_id'], q['query']
    all_sc = bm25_mirage.get_scores(qtext.lower().split())
    cand_idxs = pool_idx_by_qid[qid]
    bm25_sc   = np.array([all_sc[i] for i in cand_idxs], dtype=np.float32)
    bm25_n    = (bm25_sc - bm25_sc.min()) / (bm25_sc.max() - bm25_sc.min() + 1e-9)
    cand_emb  = chunk_emb_mpnet[cand_idxs]
    cos_n     = ((cand_emb @ qe).cpu().numpy() + 1.0) / 2.0
    mix_sc    = best_alpha * bm25_n + (1.0 - best_alpha) * cos_n
    for idx, sc in zip(cand_idxs, mix_sc):
        mix_mirage_run.append(ir_measures.ScoredDoc(qid, str(idx), float(sc)))

m_mix_mirage = ir_measures.calc_aggregate(MIRAGE_MEASURES, mirage_test_qrels, mix_mirage_run)
print(f'Mixture (alpha={best_alpha:.2f}) MIRAGE test:')
for m in MIRAGE_MEASURES:
    print(f'  {str(m):<12}{m_mix_mirage[m]:.4f}')


Mixture (alpha=0.05) MIRAGE test:
  P@1         0.6821
  nDCG@5      0.8613
  AP          0.8142


Скор = alpha * BM25_norm + (1 - alpha) * cosine_norm

BM25 нормализуем min-max по каждому запросу (по top-100 кандидатам), косинус приводим к [0,1] формулой (cos+1)/2. В качестве bi-encoder используем all-mpnet-base-v2. Оптимальный alpha ищем на WikiIR train сеткой с шагом 0.05 по nDCG@20.

Оптимальный alpha = 0.05 — практически полностью опирается на косинусное сходство bi-encoder. Даже на полном корпусе из 370k документов BM25 несёт очень мало дополнительного сигнала поверх векторов mpnet: bi-encoder уже хорошо обобщает семантику, а BM25 лишь слегка корректирует ранжирование. nDCG@20 на train: alpha=0.05 - 0.4533, alpha=0.00 - 0.4411

WikiIR test с alpha=0.05: P@1 = 0.77, P@10 = 0.265, nDCG@20 = 0.460 — лучший nDCG@20 среди заданий 1–3, лучший P@10 среди всех методов кроме fine-tuned CE

MIRAGE с alpha=0.05: P@1 = 0.682, nDCG@5 = 0.861 — выше, чем чистый mpnet (P@1 = 0.658), что показывает, что даже небольшой вклад BM25 помогает и на MIRAGE

## Дополнительное задание. Дообучение cross-encoder

Конфигурация обучения:

Базовая модель: cross-encoder/ms-marco-MiniLM-L-6-v2 (предобучена на MS MARCO, дообучаем на WikiIR).

Обучающие пары: 14 440 пар (7 220 positives + 7 220 negatives). Positives — документы из qrels с rel > 0, negatives — hard negatives из BM25 top-100 по полному корпусу. На запрос берётся до 5 пар каждого типа.

Лосс: BinaryCrossEntropy с sigmoid (num_labels=1 в CrossEncoder). Гиперпараметры: lr = 2e-5, batch_size = 32, epochs = 3, warmup_steps = 100.

Результаты на WikiIR: дообучение заметно улучшает все метрики — nDCG@20 растёт с 0.4487 до 0.4855 (+3.7pp), P@10 с 0.232 до 0.284 (+5.2pp), AP с 0.205 до 0.246 (+4.1pp). Модель адаптируется к специфике Wikipedia-поиска: hard negatives из BM25 полного корпуса — качественный обучающий сигнал.

Результаты на MIRAGE: fine-tuned CE заметно хуже pretrained — P@1 падает с 0.806 до 0.732 (-7.4pp), nDCG@5 с 0.917 до 0.882 (-3.6pp). Причина — доменный сдвиг: модель подстраивается под Wikipedia encyclopedic retrieval, тогда как MIRAGE — это медицинский фактоидный QA с совершенно другим распределением запросов и документов.

In [26]:
# Формируем обучающие пары из WikiIR training
# Positives: из qrels (rel > 0) — текст берём из полного корпуса
# Negatives: hard negatives из BM25 top-100 по полному корпусу
rng = np.random.default_rng(RNG_SEED)
ft_pairs = []

for qid, raw in tqdm(train_queries.items(), desc='build train pairs'):
    q_tokens = raw.split()
    pos_ids  = [d for d, r in train_qrels.get(qid, {}).items() if r > 0 and d in doc_id_to_pos]
    if not pos_ids:
        continue
    idx, _ = bm25_topk(q_tokens, k=100)
    neg_ids = [doc_ids[i] for i in idx if doc_ids[i] not in train_qrels.get(qid, {})]
    if not neg_ids:
        continue
    n = min(len(pos_ids), len(neg_ids), 5)
    for did in rng.choice(pos_ids, size=n, replace=False):
        text = ' '.join(docs_orig[doc_id_to_pos[did]])
        ft_pairs.append(InputExample(texts=[raw, text], label=1.0))
    for did in rng.choice(neg_ids, size=n, replace=False):
        text = ' '.join(docs_orig[doc_id_to_pos[did]])
        ft_pairs.append(InputExample(texts=[raw, text], label=0.0))

n_pos = sum(1 for x in ft_pairs if x.label == 1.0)
n_neg = sum(1 for x in ft_pairs if x.label == 0.0)
print(f'Обучающих пар: {len(ft_pairs)}  (positives: {n_pos}, negatives: {n_neg})')

build train pairs:   0%|          | 0/1444 [00:00<?, ?it/s]

Обучающих пар: 14440  (positives: 7220, negatives: 7220)


In [27]:
# Дообучение cross-encoder — запускать только на GPU
FT_BASE  = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
FT_OUT   = 'cross_encoder_finetuned'

ft_model = CrossEncoder(FT_BASE, num_labels=1, device=DEVICE, max_length=512)

train_loader = DataLoader(ft_pairs, shuffle=True, batch_size=32)

ft_model.fit(
    train_dataloader=train_loader,
    epochs=3,
    warmup_steps=100,
    optimizer_params={'lr': 2e-5},
    output_path=FT_OUT,
    show_progress_bar=True,
)
print('Дообучение завершено, модель сохранена в', FT_OUT)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
500,0.686916
1000,0.385838


Дообучение завершено, модель сохранена в cross_encoder_finetuned


In [28]:
# Оцениваем дообученную модель на WikiIR test (k=100)
run_ft_wiki, _, _ = rerank_wiki_top_k(ft_model, k=100)
m_ft_wiki = ir_measures.calc_aggregate(MEASURES, test_qrels_trec, run_ft_wiki)

print('Fine-tuned CE WikiIR test (k=100):')
for m in MEASURES:
    print(f'  {str(m):<12}{m_ft_wiki[m]:.4f}')

cmp_ft_wiki = pd.DataFrame({
    'BM25':             {str(m): bm25_wiki_metrics[m] for m in MEASURES},
    'CE pretrained k=100': {str(m): rerank_results[100]['metrics'][m] for m in MEASURES},
    'CE fine-tuned k=100': {str(m): m_ft_wiki[m] for m in MEASURES},
})
print('\nСравнение CE до и после дообучения (WikiIR):')
print(cmp_ft_wiki.round(4).to_string())


CE k=100:   0%|          | 0/100 [00:00<?, ?it/s]

Fine-tuned CE WikiIR test (k=100):
  P@1         0.8100
  P@10        0.2840
  P@20        0.1855
  AP          0.2455
  nDCG@20     0.4855

Сравнение CE до и после дообучения (WikiIR):
           BM25  CE pretrained k=100  CE fine-tuned k=100
P@1      0.4900               0.8000               0.8100
P@10     0.2120               0.2320               0.2840
P@20     0.1500               0.1665               0.1855
AP       0.1752               0.2050               0.2455
nDCG@20  0.3570               0.4487               0.4855


In [29]:
# Оцениваем дообученную модель на MIRAGE test
run_ft_mirage = rerank_mirage_ce(ft_model, k=5)
m_ft_mirage   = ir_measures.calc_aggregate(MIRAGE_MEASURES, mirage_test_qrels, run_ft_mirage)

cmp_ft_mirage = pd.DataFrame({
    'BM25':               {str(m): bm25_mirage_metrics[m] for m in MIRAGE_MEASURES},
    'CE pretrained k=5':  {str(m): mm_ce[m]       for m in MIRAGE_MEASURES},
    'CE fine-tuned k=5':  {str(m): m_ft_mirage[m] for m in MIRAGE_MEASURES},
})
print('Fine-tuned CE MIRAGE test (k=5):')
print(cmp_ft_mirage.round(4).to_string())


MIRAGE CE k=5:   0%|          | 0/1513 [00:00<?, ?it/s]

Fine-tuned CE MIRAGE test (k=5):
          BM25  CE pretrained k=5  CE fine-tuned k=5
P@1     0.5288             0.8057             0.7323
nDCG@5  0.7769             0.9173             0.8818
AP      0.7029             0.8891             0.8419


## Сводная таблица

Результаты hw2/hw3 приведены для сравнения

In [30]:
# WikiIR — сводная таблица
M = MEASURES

wiki_summary = pd.DataFrame({
    # результаты из hw2
    'BM25 original (hw2)':          dict(zip([str(m) for m in M], [0.4900, 0.2120, 0.1500, 0.1752, 0.3570])),
    'TF-IDF original (hw2)':        dict(zip([str(m) for m in M], [0.5100, 0.2030, 0.1350, 0.1655, 0.3499])),
    # результаты из hw3
    'CatBoost rerank (hw3)':         dict(zip([str(m) for m in M], [0.5400, 0.2060, 0.1425, 0.1746, 0.3676])),
    # задание 1
    'all-MiniLM-L6-v2 (task 1)':    {str(m): m_mini[m]  for m in M},
    'all-mpnet-base-v2 (task 1)':   {str(m): m_mpnet[m] for m in M},
    # задание 2
    'CE pretrained k=10 (task 2)':  {str(m): rerank_results[10]['metrics'][m]  for m in M},
    'CE pretrained k=50 (task 2)':  {str(m): rerank_results[50]['metrics'][m]  for m in M},
    'CE pretrained k=100 (task 2)': {str(m): rerank_results[100]['metrics'][m] for m in M},
    # задание 3
    f'Mixture alpha={best_alpha:.2f} (task 3)': {str(m): m_mix_wiki[m] for m in M},
    # дополнительное
    'CE fine-tuned k=100 (add)':    {str(m): m_ft_wiki[m] for m in M},
}).T

print('WikiIR test — все конфигурации:')
print(wiki_summary.round(4).to_string())


WikiIR test — все конфигурации:
                               P@1   P@10    P@20      AP  nDCG@20
BM25 original (hw2)           0.49  0.212  0.1500  0.1752   0.3570
TF-IDF original (hw2)         0.51  0.203  0.1350  0.1655   0.3499
CatBoost rerank (hw3)         0.54  0.206  0.1425  0.1746   0.3676
all-MiniLM-L6-v2 (task 1)     0.64  0.185  0.1265  0.1397   0.3574
all-mpnet-base-v2 (task 1)    0.78  0.208  0.1360  0.1713   0.4024
CE pretrained k=10 (task 2)   0.75  0.215  0.1075  0.1540   0.3759
CE pretrained k=50 (task 2)   0.81  0.235  0.1625  0.1993   0.4469
CE pretrained k=100 (task 2)  0.80  0.232  0.1665  0.2050   0.4487
Mixture alpha=0.05 (task 3)   0.77  0.265  0.1725  0.2226   0.4598
CE fine-tuned k=100 (add)     0.81  0.284  0.1855  0.2455   0.4855


In [31]:
# MIRAGE — сводная таблица
MM = MIRAGE_MEASURES

mirage_summary = pd.DataFrame({
    # hw3 (BM25 body и CatBoost из §4)
    'BM25 body (hw3)':               dict(zip([str(m) for m in MM], [0.5876, 0.8097, 0.7461])),
    'CatBoost text+pop (hw3)':        dict(zip([str(m) for m in MM], [0.6642, 0.8492, 0.7983])),
    # текущий hw4
    'BM25 (hw4 baseline)':           {str(m): bm25_mirage_metrics[m] for m in MM},
    'all-MiniLM-L6-v2 (task 1)':    {str(m): mm_mini[m]  for m in MM},
    'all-mpnet-base-v2 (task 1)':   {str(m): mm_mpnet[m] for m in MM},
    'CE pretrained k=5 (task 2)':   {str(m): mm_ce[m]    for m in MM},
    f'Mixture alpha={best_alpha:.2f} (task 3)': {str(m): m_mix_mirage[m] for m in MM},
    'CE fine-tuned k=5 (add)':      {str(m): m_ft_mirage[m] for m in MM},
}).T

print('MIRAGE test — все конфигурации:')
print(mirage_summary.round(4).to_string())


MIRAGE test — все конфигурации:
                                P@1  nDCG@5      AP
BM25 body (hw3)              0.5876  0.8097  0.7461
CatBoost text+pop (hw3)      0.6642  0.8492  0.7983
BM25 (hw4 baseline)          0.5288  0.7769  0.7029
all-MiniLM-L6-v2 (task 1)    0.6200  0.8326  0.7759
all-mpnet-base-v2 (task 1)   0.6576  0.8509  0.8002
CE pretrained k=5 (task 2)   0.8057  0.9173  0.8891
Mixture alpha=0.05 (task 3)  0.6821  0.8613  0.8142
CE fine-tuned k=5 (add)      0.7323  0.8818  0.8419


## Анализ результатов

WikiIR 100 тестовых запросов:

BM25: P@1 = 0.49, nDCG@20 = 0.357 — совпадает с hw2

Би энкодеры дают большой прирост по P@1 (MiniLM: +15, mpnet: +29), но скромный прирост по nDCG@20 (MiniLM: +0.04, mpnet: +4.5). Это объясняется природой dense ретривала: биэнкодер хорошо находит семантически близкий топ-1 документ, но на позициях 2–20 лексическая точность BM25 по-прежнему конкурентна

Cross-encoder с k=50 лучший среди pretrained моделей: P@1 = 0.81 (+32 над BM25), nDCG@20 = 0.447. При k=10 уже P@1 = 0.75, то есть даже маленький пул кандидатов даёт сильный результат. Латентность: BM25 ~370 ms/q (полный корпус), CE добавляет 24–150 ms/q в зависимости от k

Mixture alpha=0.05 — лучший nDCG@20 среди заданий 1–3: 0.460 (+10.3 над BM25), лучший P@10 = 0.265. Минимальная доля BM25 (5%) стабилизирует ранжирование хвоста.

Fine-tuned CE — лучший метод на WikiIR: nDCG@20 = 0.485, P@1 = 0.81, P@10 = 0.284. Дообучение на WikiIR train с жёсткими негативами из полного корпуса даёт значительный прирост над pretrained CE (+3.7 nDCG@20, +5.2 P@10)

MIRAGE (37 800 чанков, 1 513 запросов):

Иерархия методов: BM25 (P@1 = 0.529) < MiniLM (0.620) < mpnet (0.658) < смесь alpha=0.05 (0.682) < fine-tuned CE (0.732) < pretrained CE (0.806)

Pretrained CE — абсолютный лидер: P@1 = 0.806, nDCG@5 = 0.917. При 5 кандидатах CE может полноценно сравнить каждую пару (запрос, чанк), что даёт ему огромное преимущество перед биэнкодер

Сравнение с hw3: pretrained CE (P@1 = 0.806) значительно превосходит лучшую модель hw3 — CatBoost text+pop (P@1 = 0.664, +14.2). Биэнкодер mpnet (P@1 = 0.658) практически совпадает с CatBoost hw3 без каких-либо ручных признаков

Fine-tuning на WikiIR ухудшает MIRAGE: P@1 падает с 0.806 до 0.732 (-7.4), nDCG@5 с 0.917 до 0.882 (-3.5). Доменный сдвиг (encyclopedic Wikipedia IR → medical QA) значительно сильнее, чем выигрыш от дообучения